-------------------------------------------------------------------------
*   PONTIFÍCIA UNIVERSIDADE CATÓLICA DE MINAS GERAIS
*   PROFESSOR: VICTOR SALES SILVA
*   ALUNO: DGEISON SERRÃO PEIXOTO
*   MATRÍCULA: **1366415**
*   ATIVIDADE: LEITURA DE ARQUIVO EM FORMATO PARQUET UTILIZANDO SPARK
-------------------------------------------------------------------------

# INSTALAÇÃO DAS BIBLIOTECAS

In [ ]:
%pip install pyspark
!pip install azure-storage-blob

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 412.9/412.9 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.7/210.7 kB 16.1 MB/s eta 0:00:00


# IMPORTAÇÃO DAS BIBLIOTECAS

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import TimestampType, IntegerType, StringType, DoubleType
from pyspark.sql.functions import col
from pyspark.sql.functions import to_date
import xml.etree.ElementTree as ET
from azure.storage.blob import BlobServiceClient
from azure.storage.blob import BlobClient

# CRIAÇÃO DA APLICAÇÃO SPARK

In [ ]:
spark = SparkSession.builder.getOrCreate()

# VARIÁVEIS DE APOIO

In [ ]:
storageaccount = 'stgaccount687878'
container = 'datalake-687878'
connection_string = 'DefaultEndpointsProtocol=https;AccountName=stgaccount687878;AccountKey=SUA_ACCOUNT_KEY_AQUI;EndpointSuffix=core.windows.net'
blob_file = 'bronze/DADOS_ALUNOS/DADOS_ALUNOS.xml'

In [ ]:
def listar_arquivos_no_container(conn_string, container_name, prefixo=""):
  try:
      # 1. Conecta ao serviço de Blob
      blob_service_client = BlobServiceClient.from_connection_string(conn_string)

      # 2. Obtém o cliente para o contêiner
      container_client = blob_service_client.get_container_client(container_name)

      print(f"Buscando arquivos em '{container_name}' com o prefixo '{prefixo}'...")

      # 3. Lista os blobs (arquivos) que começam com o prefixo
      blob_list = container_client.list_blobs(name_starts_with=prefixo)

      lista_de_arquivos = []
      for blob in blob_list:
          print(f"  - {blob.name}")
          lista_de_arquivos.append(blob.name)

      if not lista_de_arquivos:
          print("\nNenhum arquivo encontrado neste caminho.")

      return lista_de_arquivos

  except Exception as e:
      print(f"Ocorreu um erro ao tentar listar os arquivos: {e}")
      return []


camada_para_verificar = 'bronze/'

print(f"--- Verificando o conteúdo da camada '{camada_para_verificar}' ---")
arquivos_encontrados = listar_arquivos_no_container(
    connection_string,
    container,
    prefixo=camada_para_verificar
)

print("\n--- Fim da verificação ---")

--- Verificando o conteúdo da camada 'bronze/' ---
Buscando arquivos em 'datalake-687878' com o prefixo 'bronze/'...
  - bronze/DADOS_ALUNOS/DADOS_ALUNOS.xml
  - bronze/DADOS_BANCARIOS/DADOS_BANCARIOS.xml
  - bronze/DADOS_ESTUDANTES/DADOS_ESTUDANTES.json
  - bronze/DADOS_EXAMES/DADOS_EXAMES.csv
  - bronze/DADOS_VOOS/DADOS_VOOS.parquet

--- Fim da verificação ---


# LEITURA DO ARQUIVO PARQUET USANDO SPARK

In [ ]:
blob_file_na_nuvem = 'bronze/DADOS_VOOS/DADOS_VOOS.parquet'
arquivo_local = 'DADOS_VOOS.parquet'

print(f"Baixando o arquivo '{blob_file_na_nuvem}' da nuvem...")
try:
    blob_client = BlobClient.from_connection_string(
        conn_str=connection_string,
        container_name=container,
        blob_name=blob_file_na_nuvem
    )
    with open(arquivo_local, "wb") as my_blob:
        blob_data = blob_client.download_blob()
        blob_data.readinto(my_blob)

    print(f"Arquivo salvo localmente como '{arquivo_local}' com sucesso!")

except Exception as e:
    print(f"Ocorreu um erro no download: {e}")
    arquivo_local = None


Baixando o arquivo 'bronze/DADOS_VOOS/DADOS_VOOS.parquet' da nuvem...
Arquivo salvo localmente como 'DADOS_VOOS.parquet' com sucesso!


In [ ]:
df_dados_voos = spark.read.format('parquet').options(header=True, inferSchema=True).load(arquivo_local)

# EXIBINDO UMA AMOSTRA DOS DADOS

In [ ]:
df_dados_voos.show()

+------+-----------+---------------+-------------+-------------------+-------------------+----------+-------------+
|origin|destination|        airline|flight_number|     departure_time|       arrival_time|passengers|delay_minutes|
+------+-----------+---------------+-------------+-------------------+-------------------+----------+-------------+
|   SDU|        SSA|            GOL|         1726|2023-07-06 11:10:10|2023-07-06 15:10:10|       145|           78|
|   REC|        FLN|           Azul|         6485|2023-05-25 01:06:09|2023-05-25 02:06:09|       107|           55|
|   FOR|        BSB|       Emirates|         4051|2023-07-14 22:20:55|2023-07-15 06:20:55|       136|           74|
|   REC|        GRU|           Azul|         8528|2023-05-30 07:45:50|2023-05-30 09:45:50|       153|          116|
|   FOR|        SDU|British Airways|         6470|2023-06-20 04:38:15|2023-06-20 08:38:15|       166|           16|
|   GRU|        REC|            GOL|         2257|2023-07-12 12:21:21|20

# EXIBINDO OS METADADOS DO ARQUIVO

In [ ]:
df_dados_voos.printSchema()

root
 |-- origin: string (nullable = true)
 |-- destination: string (nullable = true)
 |-- airline: string (nullable = true)
 |-- flight_number: long (nullable = true)
 |-- departure_time: string (nullable = true)
 |-- arrival_time: string (nullable = true)
 |-- passengers: long (nullable = true)
 |-- delay_minutes: long (nullable = true)



# AJUSTAR O SCHEMA DOS DADOS, SE NECESSÁRIO

In [ ]:
df_dados_voos = df_dados_voos.withColumn('departure_time', col('departure_time').astype(TimestampType()))
df_dados_voos = df_dados_voos.withColumn('arrival_time', col('arrival_time').astype(TimestampType()))